In [ ]:
from inputs import get_gamepad

def read_joystick():
    events = get_gamepad()
    for event in events:
        if event.ev_type == "Absolute":
            print(f"Axis {event.code} value: {event.state}")
        elif event.ev_type == "Key":
            print(f"Button {event.code} pressed: {event.state}")

while True:
    read_joystick()


In [1]:
import pygame
pygame.init()
pygame.joystick.init()
joystick = pygame.joystick.Joystick(0)
joystick.init()




<frozen importlib._bootstrap>:488: RuntimeWarning: Your system is avx2 capable but pygame was not built with support for it. The performance of some of your blits could be adversely affected. Consider enabling compile time detection with environment variables like PYGAME_DETECT_AVX2=1 if you are compiling without cross compilation.


pygame 2.5.2 (SDL 2.30.7, Python 3.12.1)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [16]:
pygame.event.pump()
print(f"Axis {0} value: {joystick.get_axis(0)}")

Axis 0 value: -1.0


In [17]:
import pygame
import time

class VehicleModel:
    def __init__(self):
        # Initialize vehicle parameters
        self.position = 0  # Simplified model, 1D position
        self.velocity = 0  # Vehicle's velocity

    def update(self, throttle, steering, dt):
        # Update position and velocity based on joystick inputs
        self.velocity += throttle * dt  # Simplified acceleration model
        self.position += self.velocity * dt  # Update position

        # Optionally, add more complex dynamics (e.g., 2D position, angular velocity)
        print(f"Position: {self.position}, Velocity: {self.velocity}")

class JoystickInterface:
    def __init__(self):
        pygame.init()
        pygame.joystick.init()
        if pygame.joystick.get_count() > 0:
            self.joystick = pygame.joystick.Joystick(0)
            self.joystick.init()
            print(f"Joystick initialized: {self.joystick.get_name()}")
        else:
            raise Exception("No joystick found")

    def get_inputs(self):
        pygame.event.pump()  # Process event queue

        # Assuming the throttle is on axis 1 and steering on axis 0
        throttle = -self.joystick.get_axis(1)  # Invert if necessary
        steering = self.joystick.get_axis(0)

        # Optionally, read button inputs if needed for other controls
        button_pressed = self.joystick.get_button(0)  # Example: button 0
        return throttle, steering, button_pressed

def main():
    vehicle = VehicleModel()
    joystick = JoystickInterface()

    # Simulation parameters
    dt = 0.05  # Time step (50 ms)
    running = True

    try:
        while running:
            throttle, steering, button = joystick.get_inputs()
            # Convert joystick inputs to model inputs (customizable)
            vehicle_throttle = throttle * 10  # Scale as needed for model
            vehicle_steering = steering       # Not used in this simple example

            # Update vehicle model
            vehicle.update(vehicle_throttle, vehicle_steering, dt)

            # Simulation delay
            time.sleep(dt)
            if button:  # Exit on button press
                running = False

    except KeyboardInterrupt:
        print("Simulation terminated by user.")
    finally:
        pygame.quit()

if __name__ == "__main__":
    main()


Joystick initialized: DualSense Wireless Controller
Position: 9.8419189453125e-05, Velocity: 0.0019683837890625
Position: 0.000295257568359375, Velocity: 0.003936767578125
Position: 0.00059051513671875, Velocity: 0.0059051513671875
Position: 0.00098419189453125, Velocity: 0.00787353515625
Position: 0.001476287841796875, Velocity: 0.0098419189453125
Position: 0.002066802978515625, Velocity: 0.011810302734375
Position: 0.0027557373046875, Velocity: 0.0137786865234375
Position: 0.0035430908203125, Velocity: 0.0157470703125
Position: 0.004428863525390625, Velocity: 0.0177154541015625
Position: 0.005413055419921875, Velocity: 0.019683837890625
Position: 0.00649566650390625, Velocity: 0.0216522216796875
Position: 0.00767669677734375, Velocity: 0.02362060546875
Position: 0.008956146240234376, Velocity: 0.0255889892578125
Position: 0.010334014892578125, Velocity: 0.027557373046875
Position: 0.011810302734375, Velocity: 0.0295257568359375
Position: 0.013385009765625, Velocity: 0.031494140625
Po

In [19]:
import pygame
import time
import numpy as np

# Define colors
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
BLUE = (0, 0, 255)

# Initialize the screen dimensions
SCREEN_WIDTH, SCREEN_HEIGHT = 800, 600
CENTER_X, CENTER_Y = SCREEN_WIDTH // 2, SCREEN_HEIGHT // 2

# 3D coordinates of a cube representing the vehicle
CUBE_VERTICES = np.array([
    [-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
    [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]
]) * 50  # Scale for visibility

CUBE_EDGES = [
    (0, 1), (1, 2), (2, 3), (3, 0),  # Back face
    (4, 5), (5, 6), (6, 7), (7, 4),  # Front face
    (0, 4), (1, 5), (2, 6), (3, 7)   # Connecting edges
]

class VehicleModel:
    def __init__(self):
        self.position = np.array([0, 0, 200])  # Start in the distance on the z-axis
        self.velocity = 0.0
        self.angle = 0.0  # Heading angle in the xy-plane

    def update(self, throttle, steering, dt):
        # Update velocity based on throttle
        acceleration = throttle * 10.0
        self.velocity += acceleration * dt
        self.position[2] -= self.velocity * dt  # Move along z-axis
        self.angle += steering * 0.05  # Steer angle (for simple yaw rotation)

    def get_transformed_vertices(self):
        # Apply rotation and translation to the cube vertices
        cos_a, sin_a = np.cos(self.angle), np.sin(self.angle)
        rotation_matrix = np.array([
            [cos_a, 0, sin_a],
            [0, 1, 0],
            [-sin_a, 0, cos_a]
        ])
        transformed_vertices = (rotation_matrix @ CUBE_VERTICES.T).T + self.position
        return transformed_vertices

class JoystickInterface:
    def __init__(self):
        pygame.init()
        pygame.joystick.init()
        self.joystick = None
        if pygame.joystick.get_count() > 0:
            self.joystick = pygame.joystick.Joystick(0)
            self.joystick.init()
            print(f"Joystick initialized: {self.joystick.get_name()}")
        else:
            raise Exception("No joystick found")

    def get_inputs(self):
        pygame.event.pump()  # Process the event queue

        # Assuming axes 1 and 0 for throttle and steering
        throttle = -self.joystick.get_axis(1)  # Throttle
        steering = self.joystick.get_axis(0)   # Steering

        return throttle, steering

def project_to_2d(point):
    """ Project 3D point onto 2D surface with simple perspective projection """
    focal_length = 300
    x, y, z = point
    if z != 0:  # Prevent division by zero
        x_2d = int(CENTER_X + (x * focal_length) / z)
        y_2d = int(CENTER_Y - (y * focal_length) / z)
        return (x_2d, y_2d)
    return (CENTER_X, CENTER_Y)

def draw_cube(screen, vertices):
    """ Draw the cube by connecting edges """
    for edge in CUBE_EDGES:
        start, end = edge
        start_pos = project_to_2d(vertices[start])
        end_pos = project_to_2d(vertices[end])
        pygame.draw.line(screen, BLUE, start_pos, end_pos, 2)

def simulate():
    screen = pygame.display.set_mode((SCREEN_WIDTH, SCREEN_HEIGHT))
    pygame.display.set_caption("3D Vehicle Simulation")
    clock = pygame.time.Clock()

    vehicle = VehicleModel()
    joystick = JoystickInterface()

    running = True
    try:
        while running:
            # Handle events
            for event in pygame.event.get():
                if event.type == pygame.QUIT:
                    running = False

            # Get joystick inputs
            throttle, steering = joystick.get_inputs()

            # Update vehicle state based on joystick inputs
            vehicle.update(throttle, steering, dt=0.05)

            # Get transformed cube vertices
            vertices = vehicle.get_transformed_vertices()

            # Clear screen
            screen.fill(WHITE)

            # Draw the 3D cube as vehicle
            draw_cube(screen, vertices)

            # Update display
            pygame.display.flip()

            # Cap the frame rate
            clock.tick(60)  # 20 FPS

    except KeyboardInterrupt:
        print("Simulation interrupted by user.")
    finally:
        pygame.quit()

if __name__ == "__main__":
    simulate()


Joystick initialized: DualSense Wireless Controller
